# 02. Auditoría de identidad de las imágenes: regla, revisión manual y grupos finales

**Regla automática (secuencia de captura).** Dos imágenes del conjunto principal quedan en el mismo grupo si cumplen cualquiera de estas condiciones:

- pHash ≤ 6 o coseno DINOv2 ≥ 0.95;
- misma carpeta, números a distancia ≤ 3 y coseno ≥ 0.75 (fotos sucesivas de la misma mazorca);
- ≥ 40 correspondencias SIFT + RANSAC (misma escena);
- coseno ≥ 0.93 aunque los números estén lejos.

**Revisión manual.** Los pares que la regla no unió pero tienen alguna evidencia (números a distancia 4 a 10 con coseno ≥ 0.80, coseno 0.88 a 0.93, o 20 a 39 correspondencias SIFT) se revisan uno por uno. La herramienta solo muestra las dos fotografías: ni nombres de archivo, ni clases, ni predicciones. "No sé" se trata como "misma" (criterio conservador); los pares que no se revisen se tratan como distintos. En el estudio se revisaron 396 de los 650 pares (decisiones en `auditoria_identidad/revision_decisiones.csv`). Las respuestas se guardan en cada clic y la revisión puede retomarse.

**Sanas no usadas.** Se marcan dos niveles de vínculo con el conjunto principal: por la regla automática (se excluyen en el análisis principal de la evaluación complementaria) y por la franja dudosa (se excluyen además en un análisis de sensibilidad).

Ejecuta la celda 1; la celda 2 abre la herramienta de revisión; al terminar la revisión, ejecuta la celda 3.


## 1. Regla automática y pares pendientes

In [ ]:
import os, io, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as W
from IPython.display import display

PROJECT_DIR = Path.cwd()
A4 = PROJECT_DIR / 'auditoria_identidad'
DECISIONES = A4 / 'revision_decisiones.csv'
REGLA = dict(ham_max=6, cos_alto=0.95, dnum_seq=3, cos_seq=0.75, inliers_escena=40, cos_lejos=0.93)
DUDOSA = dict(dnum_min=4, dnum_max=10, cos_dnum=0.80, cos_lo=0.88, cos_hi=0.93, inl_lo=20, inl_hi=39)

def ruta_local(p):
    return str(PROJECT_DIR / p)

im = pd.read_csv(A4 / 'imagenes.csv', index_col='id')
im['cache_path'] = im['cache_path'].apply(ruta_local)
p = pd.read_csv(A4 / 'pares_candidatos.csv')
F = np.load(A4 / 'feats_dinov2.npy')
main = (im['conjunto'] == 'principal').values

def regla(q):
    return ((q['ham'] <= REGLA['ham_max']) | (q['cos'] >= REGLA['cos_alto']) | (q['inliers'] >= REGLA['inliers_escena'])
            | (q['cos'] >= REGLA['cos_lejos']) | (q['dnum'].between(1, REGLA['dnum_seq']) & (q['cos'] >= REGLA['cos_seq'])))

def dudosa(q):
    return ((q['dnum'].between(DUDOSA['dnum_min'], DUDOSA['dnum_max']) & (q['cos'] >= DUDOSA['cos_dnum']))
            | q['cos'].between(DUDOSA['cos_lo'], DUDOSA['cos_hi']) | q['inliers'].between(DUDOSA['inl_lo'], DUDOSA['inl_hi']))

class UF:
    def __init__(s, n): s.p = list(range(n))
    def f(s, x):
        while s.p[x] != x:
            s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def u(s, a, b):
        ra, rb = s.f(a), s.f(b)
        if ra != rb: s.p[rb] = ra
    def grupos(s): return np.array([s.f(i) for i in range(len(s.p))])

pm = p[main[p['i']] & main[p['j']]].copy()
pm['por_regla'] = regla(pm)
uf = UF(len(im))
for a, b in zip(pm.loc[pm['por_regla'], 'i'], pm.loc[pm['por_regla'], 'j']):
    uf.u(a, b)
g_auto = uf.grupos()
pend = pm[~pm['por_regla'] & dudosa(pm)].copy()
pend = pend[g_auto[pend['i']] != g_auto[pend['j']]]
pend = pend.sample(frac=1, random_state=2024).reset_index(drop=True)      # orden aleatorio fijo
pend[['i', 'j', 'cos', 'ham', 'dnum', 'inliers']].to_csv(A4 / 'revision_pares_pendientes.csv', index=False)
s = pd.Series(g_auto[main]).value_counts()
print('Regla automática: %d grupos en el conjunto principal | el mayor tiene %d imágenes' % (len(s), s.max()))
print('Pares de la franja dudosa para revisar:', len(pend))


## 2. Revisión manual (opcional; se guarda en cada clic)

In [ ]:
# Herramienta de revisión. Las decisiones se guardan en auditoria_identidad/revision_decisiones.csv
LADO = 430
if DECISIONES.exists():
    dec = pd.read_csv(DECISIONES)
else:
    dec = pd.DataFrame(columns=['i', 'j', 'decision', 'hora'])
hechos = {(int(a), int(b)) for a, b in zip(dec['i'], dec['j'])}
cola = [k for k in range(len(pend)) if (int(pend.at[k, 'i']), int(pend.at[k, 'j'])) not in hechos]
estado = {'pos': 0, 'historial': []}

def png(path):
    b = io.BytesIO()
    Image.open(path).convert('RGB').resize((LADO, LADO)).save(b, format='PNG')
    return b.getvalue()

img_a, img_b = W.Image(width=LADO, height=LADO), W.Image(width=LADO, height=LADO)
info = W.HTML()
b_misma = W.Button(description='Misma mazorca o escena', button_style='danger', layout=W.Layout(width='220px'))
b_dist = W.Button(description='Distinta', button_style='success', layout=W.Layout(width='140px'))
b_nose = W.Button(description='No sé', button_style='warning', layout=W.Layout(width='100px'))
b_atras = W.Button(description='Atrás', layout=W.Layout(width='100px'))

def mostrar():
    total = len(pend)
    if estado['pos'] >= len(cola):
        info.value = '<b>Revisión terminada: %d de %d pares.</b> Ejecuta la celda 3.' % (total, total)
        img_a.value = img_b.value = b''
        return
    k = cola[estado['pos']]
    i, j = int(pend.at[k, 'i']), int(pend.at[k, 'j'])
    img_a.value, img_b.value = png(ruta_local(im.at[i, 'image_path'])), png(ruta_local(im.at[j, 'image_path']))
    hechos_ahora = total - len(cola) + estado['pos']
    info.value = '<b>Par %d de %d</b> (pendientes en esta sesión: %d)' % (hechos_ahora + 1, total, len(cola) - estado['pos'])

def guardar(d):
    global dec
    if estado['pos'] >= len(cola):
        return
    k = cola[estado['pos']]
    fila = {'i': int(pend.at[k, 'i']), 'j': int(pend.at[k, 'j']), 'decision': d, 'hora': time.strftime('%Y-%m-%d %H:%M:%S')}
    dec = pd.concat([dec[~((dec['i'] == fila['i']) & (dec['j'] == fila['j']))], pd.DataFrame([fila])], ignore_index=True)
    dec.to_csv(DECISIONES, index=False)
    estado['historial'].append(estado['pos'])
    estado['pos'] += 1
    mostrar()

def atras(_):
    if estado['historial']:
        estado['pos'] = estado['historial'].pop()
        mostrar()

b_misma.on_click(lambda _: guardar('misma'))
b_dist.on_click(lambda _: guardar('distinta'))
b_nose.on_click(lambda _: guardar('no_se'))
b_atras.on_click(atras)
mostrar()
display(W.VBox([info, W.HBox([img_a, img_b]), W.HBox([b_misma, b_dist, b_nose, b_atras])]))


## 3. Grupos finales

In [ ]:
# Grupos finales
dec = pd.read_csv(DECISIONES) if DECISIONES.exists() else pd.DataFrame(columns=['i', 'j', 'decision'])
falta = len(pend) - len(dec.merge(pend[['i', 'j']], on=['i', 'j']))
print('Pares revisados: %d de %d; los %d no revisados se tratan como distintos.' % (len(pend) - falta, len(pend), falta))
unir = dec[dec['decision'].isin(['misma', 'no_se'])][['i', 'j']]
uf = UF(len(im))
for a, b in zip(pm.loc[pm['por_regla'], 'i'], pm.loc[pm['por_regla'], 'j']):
    uf.u(a, b)
for a, b in zip(unir['i'], unir['j']):
    uf.u(int(a), int(b))
g = uf.grupos()
print(dec['decision'].value_counts().rename({'misma': 'misma mazorca o escena', 'distinta': 'distinta', 'no_se': 'no sé'}).to_string())

gm = pd.DataFrame({'image_path': im.loc[main, 'image_path'], 'label': im.loc[main, 'label'], 'group': g[main]})
tam = gm.groupby('group').size()
mix = gm.groupby('group')['label'].nunique()
print('\nGrupos: %d | imágenes en grupos de más de una: %d | el mayor: %d | grupos con ambas clases: %d' % (
    len(tam), int(tam[tam > 1].sum()), tam.max(), int((mix > 1).sum())))
# Sensibilidad: tamaño de los grupos si "no sé" se tratara como "distinta"
uf2 = UF(len(im))
for a, b in zip(pm.loc[pm['por_regla'], 'i'], pm.loc[pm['por_regla'], 'j']):
    uf2.u(a, b)
for a, b in zip(dec.loc[dec['decision'] == 'misma', 'i'], dec.loc[dec['decision'] == 'misma', 'j']):
    uf2.u(int(a), int(b))
t2 = pd.Series(uf2.grupos()[main]).value_counts()
print('Si "no sé" contara como distinta: %d grupos, el mayor con %d imágenes' % (len(t2), t2.max()))
if tam.max() > 0.2 * main.sum():
    print('AVISO: el grupo mayor supera el 20 %% de las imágenes (%d); la validación cruzada de 5 pliegues quedaría '
          'desbalanceada. Revisar las decisiones "misma" que encadenan escenas antes de continuar.' % tam.max())
(PROJECT_DIR / 'grupos').mkdir(exist_ok=True)
gm.to_csv(PROJECT_DIR / 'grupos' / 'grupos_finales.csv', index=False)

# Sanas no usadas: excluidas si tienen vínculo (regla o franja dudosa) con el conjunto principal
x = p[main[p['i']] ^ main[p['j']]]
lado_nu = lambda q: set(np.where(~main[q['i']], q['i'], q['j']))
v_regla, v_dudosa = lado_nu(x[regla(x)]), lado_nu(x[dudosa(x)])
nu = im.loc[~main, ['image_path', 'fuente', 'num']].copy()
nu['vinculo_regla'] = [k in v_regla for k in nu.index]
nu['vinculo_dudoso'] = [k in v_dudosa for k in nu.index]
nu.to_csv(A4 / 'sanas_no_usadas_vinculos.csv', index=False)
print('Sanas no usadas vinculadas al conjunto principal: por la regla %d; por la regla o la franja dudosa %d (de %d)' % (
    nu['vinculo_regla'].sum(), (nu['vinculo_regla'] | nu['vinculo_dudoso']).sum(), len(nu)))

# Censo de la escena "mazorca en la mano sobre un recipiente" (centroide DINOv2 de black_pod_rot_499 a 513)
ref = im[(im['carpeta'] == 'bpr') & im['num'].between(499, 513) & main].index
c = F[ref].mean(0); c /= np.linalg.norm(c)
im['sim_escena_cajon'] = F @ c
censo = im[im['sim_escena_cajon'] >= 0.80]
print('\nEscena mano + recipiente (similitud >= 0.80 con el centroide):')
display(censo.groupby(['conjunto', 'label']).size().rename('imágenes').to_frame())
im[['image_path', 'conjunto', 'label', 'num', 'sim_escena_cajon']].to_csv(A4 / 'censo_escena_cajon.csv')

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(tam.values, bins=range(1, tam.max() + 2), color='grey')
ax.set_yscale('log'); ax.set_xlabel('Imágenes por grupo'); ax.set_ylabel('Grupos')
plt.tight_layout(); fig.savefig(A4 / 'tamano_grupos.png', dpi=200); plt.show()
json.dump({'regla': REGLA, 'franja_dudosa': DUDOSA, 'pares_revisados': int(len(dec)),
           'decisiones': dec['decision'].value_counts().to_dict(),
           'grupos': int(len(tam)), 'grupo_mayor': int(tam.max()), 'grupos_mixtos': int((mix > 1).sum()),
           'no_usadas_vinculo_regla': int(nu['vinculo_regla'].sum()), 'no_usadas_vinculo_regla_o_dudoso': int((nu['vinculo_regla'] | nu['vinculo_dudoso']).sum())},
          open(A4 / 'resumen_grupos.json', 'w'), indent=2, ensure_ascii=False)
print('Guardado: grupos/grupos_finales.csv y, en auditoria_identidad/, sanas_no_usadas_vinculos.csv, censo_escena_cajon.csv y resumen_grupos.json')
